# 06 — Loss, Chain Rule, Backpropagation, and Gradient Checking (Manual NumPy)


> **Learning contract.** Every code cell is preceded by an explanation of what the code does, why the operation exists mathematically, what tensor/array shapes are expected, and what production or business failure it prevents. Run the notebooks in numerical order in a fresh Conda environment.


Loss converts prediction quality into an optimization objective. For one correct class with predicted probability $p_y$, cross-entropy is $L=-\log p_y$. Backpropagation does **not** "send the error backward" as a number; it applies the chain rule to compute each parameter's sensitivity $\partial L/\partial \theta$.

For a sigmoid scalar example $z=wx+b$, $a=\sigma(z)$ and BCE with target 1, $\partial L/\partial z=a-1$, so $\partial L/\partial w=(a-1)x$.


## Code walkthrough — manual derivative and finite-difference verification
The analytical derivative is compared to the central-difference approximation. Gradient checking is slow and not used for training, but it is a powerful debugging technique for custom layers and manual backprop implementations.


In [1]:
import numpy as np

x = 0.8
w = 0.5
b = -0.15
y = 1.0


def scalar_loss(w_value):
    z = w_value * x + b
    a = 1 / (1 + np.exp(-z))
    return -(y * np.log(a) + (1 - y) * np.log(1 - a))


z = w * x + b
a = 1 / (1 + np.exp(-z))
analytic = (a - y) * x
eps = 1e-05
numeric = (scalar_loss(w + eps) - scalar_loss(w - eps)) / (2 * eps)
print("z", z, "a", a, "loss", scalar_loss(w))
print(
    "analytic dL/dw",
    analytic,
    "finite-difference",
    numeric,
    "absolute error",
    abs(analytic - numeric),
)

z 0.25 a 0.5621765008857981 loss 0.5759394198788436
analytic dL/dw -0.35025879929136156 finite-difference -0.350258799286518 absolute error 4.8435699895321704e-12


## Code walkthrough — map the same derivative into Manual NumPy
This cell shows what the framework automates. The local mathematics is unchanged; only gradient bookkeeping changes. Inspect the resulting gradient values instead of treating `backward()` or `GradientTape` as magic.


In [2]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(0)
X = rng.normal(size=(5, 3))
y = np.array([0, 1, 0, 1, 1])
W1 = rng.normal(0, 0.2, size=(3, 4))
b1 = np.zeros((1, 4))
W2 = rng.normal(0, 0.2, size=(4, 2))
b2 = np.zeros((1, 2))
z1 = X @ W1 + b1
a1 = np.maximum(z1, 0)
logits = a1 @ W2 + b2
exp = np.exp(logits - logits.max(1, keepdims=True))
p = exp / exp.sum(1, keepdims=True)
dz2 = p.copy()
dz2[np.arange(len(X)), y] -= 1
dz2 /= len(X)
dW2 = a1.T @ dz2
db2 = dz2.sum(0, keepdims=True)
da1 = dz2 @ W2.T
dz1 = da1 * (z1 > 0)
dW1 = X.T @ dz1
print(
    "dW1",
    dW1.shape,
    "dW2",
    dW2.shape,
    "gradient norms",
    np.linalg.norm(dW1),
    np.linalg.norm(dW2),
)

dW1 (3, 4) dW2 (4, 2) gradient norms 0.11022099930985421 0.04643805235039073


## Chain-rule map for the two-layer MLP
`loss → dLogits → dW2/db2 → dA1 → dZ1 through activation derivative → dW1/db1`. Every arrow multiplies or contracts a local derivative with an upstream gradient. Shape checking is one of the best ways to debug backprop.


## Business implication
A model can produce plausible predictions while gradients are wrong, especially in custom objectives or layers. Gradient verification and loss sanity checks reduce the risk of silently training the wrong objective.
